In [ ]:
import os

!git clone https://github.com/aloshdenny/reverse-SynthID.git
%cd reverse-SynthID

# The virtual environment creation failed, and requirements were installed directly into Colab's environment.
# Removing the venv creation for simplicity in Colab, as dependencies are already satisfied.
!pip install -r requirements.txt

Cloning into 'reverse-SynthID'...
remote: Enumerating objects: 1750, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 1750 (delta 5), reused 4 (delta 2), pack-reused 1734 (from 2)
Receiving objects: 100% (1750/1750), 1.31 GiB | 45.12 MiB/s, done.
Resolving deltas: 100% (379/379), done.
Updating files: 100% (1599/1599), done.
/content/reverse-SynthID
Error: Command '['/content/reverse-SynthID/venv/bin/python3', '-m', 'ensurepip', '--upgrade', '--default-pip']' returned non-zero exit status 1.


In [ ]:
!python src/extraction/synthid_bypass.py build-codebook \
    --black gemini_black \
    --white gemini_white \
    --watermarked gemini_random \
    --output artifacts/spectral_codebook_v3.npz

[black] 100 images  resolution=(1024, 1024)
  10/100
  20/100
  30/100
  40/100
  50/100
  60/100
  70/100
  80/100
  90/100
  100/100
  mean consistency (G)=0.6436
[white] 100 images
  cross-validated bins (|cos|>0.90, G): 300954
  Top-10 phase-consistent carriers (G):
    (  +9,  -9)  mag=    80879  cons=1.0000  agree=1.000
    (  -9,  +9)  mag=    80879  cons=1.0000  agree=1.000
    ( -14,  +5)  mag=    75041  cons=1.0000  agree=0.665
    ( +14,  -5)  mag=    75041  cons=1.0000  agree=0.665
    (  -5,  -5)  mag=    96513  cons=1.0000  agree=0.993
    (  +5,  +5)  mag=    96513  cons=1.0000  agree=0.993
    ( +13,  +6)  mag=    75941  cons=1.0000  agree=0.821
    ( -13,  -6)  mag=    75941  cons=1.0000  agree=0.821
    ( -10, -11)  mag=    73527  cons=1.0000  agree=0.997
    ( +10, +11)  mag=    73527  cons=1.0000  agree=0.997

Profile added: 1024x1024  (100b+100w+0r refs)
[watermarked] 88 images  resolution=(1536, 2816)
  10/88
  20/88
  30/88
  40/88
  50/88
  60/88
  70/88
  80/88

In [ ]:
from src.extraction.synthid_bypass import SynthIDBypass, SpectralCodebook
from PIL import Image
import numpy as np
import os
import glob

codebook = SpectralCodebook()
codebook.load('artifacts/spectral_codebook_v3.npz')

# Dynamically find the first PNG image in the gemini_random directory
search_path = 'gemini_random/*.png'
available_images = glob.glob(search_path)

if not available_images:
    print(f"Error: No PNG images found in ./gemini_random/")
    !ls -R ./gemini_random
    raise FileNotFoundError("No images available to process.")

image_path = available_images[0]
print(f"Processing image: {image_path}")

# Open the image using PIL
image_pil = Image.open(image_path).convert('RGB')
# Convert to a NumPy array
image_rgb = np.array(image_pil)

bypass = SynthIDBypass()
result = bypass.bypass_v3(image_rgb, codebook, strength='aggressive')

print(f"\nPSNR: {result.psnr:.1f} dB")
print(f"Profile used: {result.details['profile_resolution']}")
print(f"Exact match: {result.details['exact_match']}")

Codebook loaded: [1024x1024, 1536x2816]
  1024x1024: 100b+100w+0r
  1536x2816: 0b+0w+88r
Processing image: gemini_random/Gemini_Generated_Image_i1d1p2i1d1p2i1d1.png

PSNR: 44.2 dB
Profile used: 1536x2816
Exact match: True


In [ ]:
# Using an existing image from the dataset for detection
!python src/extraction/robust_extractor.py detect gemini_random/Gemini_Generated_Image_i1d1p2i1d1p2i1d1.png \
    --codebook artifacts/spectral_codebook_v3.npz